# Advanced Recommender — R&D Notebook

Cel: zastąpienie TF-IDF na gatunkach → **LLM Embeddings na fabule** oraz wzbogacenie XAI o **wartości SHAP**.

Wynik tego notatnika posłuży jako draft do przepisania `recommender_engine.py` (v2).

## Architektura
```
movies.csv + links.csv
       ↓
TMDB API → overview (fabuła)
       ↓
sentence-transformers → embeddingi (384-wymiarowe wektory)
       ↓
Cosine Similarity (CB) + CF (jak dotychczas)
       ↓
Hybrid Score
       ↓
SHAP → wpływ konkretnych cech na wynik
       ↓
Struktura zgodna z kontraktem API
```

## 0. Weryfikacja środowiska

Upewnij się że pracujesz na izolowanym venv z `notebooks/requirements.txt`, a nie na środowisku backendu.

In [1]:
import sys
print(f"Python: {sys.version}")

required = ["sentence_transformers", "shap", "pandas", "sklearn", "numpy", "requests"]
missing = []
for pkg in required:
    try:
        __import__(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg} — BRAK")
        missing.append(pkg)

if missing:
    print(f"\nZainstaluj brakujące pakiety: pip install {' '.join(missing)}")
else:
    print("\nŚrodowisko OK — można zaczynać.")

Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)]


c:\Users\domar\RecoFlix\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✅ sentence_transformers
  ✅ shap
  ✅ pandas
  ✅ sklearn
  ✅ numpy
  ✅ requests

Środowisko OK — można zaczynać.


## 1. Wczytanie danych MovieLens

In [2]:
import pandas as pd
import numpy as np

DATA_DIR = "../data/ml-100k/ml-latest-small"

movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")
links = pd.read_csv(f"{DATA_DIR}/links.csv")

# tmdbId może być float (NaN) — konwertujemy na Int64 (nullable integer)
links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce").astype("Int64")

# Łączymy movies z links żeby mieć tmdbId przy każdym filmie
movies = movies.merge(links[["movieId", "tmdbId"]], on="movieId", how="left")

print(f"Filmy: {len(movies)}, Oceny: {len(ratings)}")
display(movies.head())

Filmy: 9125, Oceny: 100004


,movieId,title,genres,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844
2,3,Grumpier Old Men (1995),Comedy|Romance,15602
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357
4,5,Father of the Bride Part II (1995),Comedy,11862


## 2. Pobranie opisów fabuły z TMDB

Odpytujemy TMDB API i zapisujemy wyniki do lokalnego cache (`overviews_cache.json`).  
Przy kolejnym uruchomieniu komórki cache jest odczytywany — zero dodatkowych zapytań.

Uzupełnij swój klucz TMDB poniżej (ten sam co w `backend/.env`).

In [ ]:
import requests
import json
import time
import os
from pathlib import Path

TMDB_API_KEY = os.getenv("TMDB_API_KEY")
TMDB_LANGUAGE = "pl-PL"  # opisy po polsku
CACHE_PATH = Path("overviews_cache.json")
TMDB_BASE = "https://api.themoviedb.org/3/movie"

# Wczytaj istniejący cache
if CACHE_PATH.exists():
    with CACHE_PATH.open("r", encoding="utf-8") as f:
        cache: dict = json.load(f)
    print(f"Wczytano cache: {len(cache)} wpisów.")
else:
    cache = {}
    print("Brak cache — zostanie utworzony.")


def fetch_overview(tmdb_id: int) -> str | None:
    key = str(tmdb_id)
    if key in cache:
        return cache[key]

    if not TMDB_API_KEY:
        return None

    try:
        r = requests.get(
            f"{TMDB_BASE}/{tmdb_id}",
            params={"api_key": TMDB_API_KEY, "language": TMDB_LANGUAGE},
            timeout=5,
        )
        if r.status_code == 200:
            overview = r.json().get("overview") or None
            cache[key] = overview
            return overview
    except Exception:
        pass
    return None


# Pobieramy opisy — z rate limitingiem żeby nie przeciążyć TMDB (40 req/s limit)
overviews = []
new_fetches = 0

for _, row in movies.iterrows():
    tmdb_id = row["tmdbId"]
    if pd.isna(tmdb_id):
        overviews.append(None)
        continue

    overview = fetch_overview(int(tmdb_id))
    overviews.append(overview)

    if str(int(tmdb_id)) not in cache or cache[str(int(tmdb_id))] is None:
        new_fetches += 1
    if new_fetches > 0 and new_fetches % 40 == 0:
        time.sleep(1)  # krótka przerwa co 40 nowych zapytań

movies["overview"] = overviews

# Zapisz zaktualizowany cache
with CACHE_PATH.open("w", encoding="utf-8") as f:
    json.dump(cache, f, ensure_ascii=False)

has_overview = movies["overview"].notna().sum()
print(f"Filmy z opisem fabuły: {has_overview} / {len(movies)} ({has_overview/len(movies)*100:.1f}%)")

Brak cache — zostanie utworzony.


## 3. Przygotowanie tekstu wejściowego dla embeddingów

Łączymy fabuły i gatunki w jeden string. Dla filmów bez opisu fallback to same gatunki — tak samo jak robi to obecny model TF-IDF.

In [ ]:
def build_text(row: pd.Series) -> str:
    genres = row["genres"].replace("|", " ") if pd.notna(row["genres"]) else ""
    overview = row["overview"] if pd.notna(row["overview"]) else ""
    # overview ma priorytet; genres zawsze doklejamy na końcu żeby model wiedział o gatunku
    parts = [p for p in [overview, genres] if p]
    return " ".join(parts)


movies["text"] = movies.apply(build_text, axis=1)

print("Przykładowe teksty wejściowe:")
for _, row in movies.head(3).iterrows():
    print(f"\n[{row['title']}]")
    print(f"  {row['text'][:200]}..." if len(row['text']) > 200 else f"  {row['text']}")

## 4. LLM Embeddingi — sentence-transformers

Model `all-MiniLM-L6-v2` jest darmowy, lokalny i szybki — dobre rozwiązanie na start.  
Tworzy wektory 384-wymiarowe. Embeddingi zapisujemy do `.npy` żeby nie przeliczać przy każdym restarcie.

> **Uwaga:** pierwsze uruchomienie pobiera model (~80MB). Kolejne są natychmiastowe.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

EMBEDDINGS_PATH = Path("embeddings_minilm.npy")
MODEL_NAME = "all-MiniLM-L6-v2"
# Interfejs przygotowany do podmiany na OpenAI:
# MODEL_NAME = "text-embedding-3-small"  # ← odkomentuj i dostosuj fetch jeśli chcesz OpenAI

if EMBEDDINGS_PATH.exists():
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f"Wczytano embeddingi z cache: {embeddings.shape}")
else:
    print(f"Generuję embeddingi modelem {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME)
    texts = movies["text"].tolist()
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"Zapisano embeddingi: {embeddings.shape}")

# Macierz podobieństwa Content-Based (embeddingi zamiast TF-IDF)
cosine_sim_cb = cosine_similarity(embeddings, embeddings)
print(f"\nMacierz CB (embedding): {cosine_sim_cb.shape}")

### Porównanie: TF-IDF vs Embeddingi

Zobaczmy czy nowy model daje lepsze wyniki niż stary na konkretnym przykładzie.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Stary model — TF-IDF na gatunkach
tfidf = TfidfVectorizer(token_pattern=r"[a-zA-Z0-9\-]+")
tfidf_matrix = tfidf.fit_transform(movies["genres"].fillna(""))
cosine_sim_tfidf = cosine_similarity(tfidf_matrix, tfidf_matrix)


def compare_models(target_title: str, top_n: int = 5):
    try:
        idx = movies[movies["title"].str.contains(target_title, case=False, na=False, regex=False)].index[0]
    except IndexError:
        print(f"Nie znaleziono: {target_title}")
        return

    title = movies["title"].iloc[idx]
    print(f"Film bazowy: {title}")
    print(f"{'Film':<45} {'TF-IDF':>8} {'Embedding':>10}")
    print("-" * 65)

    tfidf_scores = sorted(enumerate(cosine_sim_tfidf[idx]), key=lambda x: x[1], reverse=True)[1:top_n+1]
    emb_scores = sorted(enumerate(cosine_sim_cb[idx]), key=lambda x: x[1], reverse=True)[1:top_n+1]

    tfidf_titles = {movies["title"].iloc[i]: s for i, s in tfidf_scores}
    emb_titles = {movies["title"].iloc[i]: s for i, s in emb_scores}

    all_titles = set(tfidf_titles) | set(emb_titles)
    for t in sorted(all_titles, key=lambda x: emb_titles.get(x, 0), reverse=True):
        tf = f"{tfidf_titles.get(t, 0):.3f}" if t in tfidf_titles else "  -  "
        em = f"{emb_titles.get(t, 0):.3f}" if t in emb_titles else "  -  "
        print(f"{t:<45} {tf:>8} {em:>10}")


compare_models("Toy Story")
print()
compare_models("Matrix")

## 5. Collaborative Filtering — bez zmian względem V1

In [ ]:
user_item_matrix = ratings.pivot(index="movieId", columns="userId", values="rating").fillna(0)
user_item_matrix = user_item_matrix.reindex(movies["movieId"], fill_value=0)
cosine_sim_cf = cosine_similarity(user_item_matrix)

print(f"Macierz CF: {cosine_sim_cf.shape}")

## 6. Zaawansowane XAI — SHAP

### Jak to działa

SHAP traktuje każdą **cechę** (gatunek, dekada, średnia ocena itp.) jako "gracza" i mierzy jej wkład w wynik rekomendacji.  
Używamy `shap.KernelExplainer` — działa z dowolną funkcją, więc nie musimy zmieniać modelu.

Podejście:
1. Budujemy prosty DataFrame cech dla każdego filmu (gatunki jako kolumny binarne + dekada + średnia ocena)
2. Definiujemy funkcję predykcji: *"jak bardzo film X pasuje do profilu użytkownika"*
3. KernelExplainer liczy wartości SHAP dla każdej cechy
4. Wynik: wiemy które cechy **najbardziej wpłynęły** na konkretną rekomendację

In [ ]:
import shap
import re

# Budujemy macierz cech — gatunki binarne + dekada + średnia ocena
all_genres = set()
for g in movies["genres"].dropna():
    all_genres.update(g.split("|"))
all_genres.discard("(no genres listed)")
genre_cols = sorted(all_genres)

features = pd.DataFrame(index=movies.index)

for genre in genre_cols:
    features[f"genre_{genre}"] = movies["genres"].fillna("").apply(lambda g: int(genre in g))

# Dekada (z tytułu — MovieLens trzyma rok w nawiasie)
def extract_decade(title: str) -> int:
    match = re.search(r"\((\d{4})\)", title)
    if match:
        return (int(match.group(1)) // 10) * 10
    return 0

features["decade"] = movies["title"].apply(extract_decade)

# Średnia ocena z ratings
avg_ratings = ratings.groupby("movieId")["rating"].mean().rename("avg_rating")
features["avg_rating"] = movies["movieId"].map(avg_ratings).fillna(3.0)

features_array = features.values.astype(float)
feature_names = features.columns.tolist()

print(f"Macierz cech: {features_array.shape}")
print(f"Cechy: {len(genre_cols)} gatunków + dekada + avg_rating = {len(feature_names)} łącznie")

In [ ]:
def get_similarity_scores(candidate_features: np.ndarray, base_idx: int, alpha: float = 0.5) -> np.ndarray:
    """
    Funkcja predykcji dla SHAP.
    Dla każdego kandydata (wiersza w candidate_features) zwraca wynik hybrydowy.
    SHAP będzie wyłączał kolejne cechy i sprawdzał jak zmienia się wynik — stąd wie co jest ważne.
    """
    scores = []
    for candidate in candidate_features:
        # Similarity cech — dot product znormalizowany (przybliżenie cosine)
        base = features_array[base_idx]
        norm_b = np.linalg.norm(base)
        norm_c = np.linalg.norm(candidate)
        if norm_b > 0 and norm_c > 0:
            cb_sim = np.dot(base, candidate) / (norm_b * norm_c)
        else:
            cb_sim = 0.0

        # CF score dla tego kandydata bierzemy z gotowej macierzy
        # (SHAP nie modyfikuje CF — modyfikuje tylko cechy CB)
        candidate_idx = np.where((features_array == candidate).all(axis=1))[0]
        cf_sim = cosine_sim_cf[base_idx, candidate_idx[0]] if len(candidate_idx) > 0 else 0.0

        scores.append(alpha * cb_sim + (1 - alpha) * cf_sim)
    return np.array(scores)


def explain_recommendation(target_title: str, alpha: float = 0.5, top_n: int = 3, shap_samples: int = 50):
    """
    Generuje rekomendacje z wartościami SHAP dla każdej rekomendacji.
    shap_samples: ile próbek używa KernelExplainer — więcej = dokładniej, wolniej.
    """
    try:
        idx = movies[movies["title"].str.contains(target_title, case=False, na=False, regex=False)].index[0]
    except IndexError:
        print(f"Nie znaleziono: {target_title}")
        return

    base_title = movies["title"].iloc[idx]
    print(f"Film bazowy: {base_title}\n")

    # Hybrid scores dla wszystkich filmów
    cb_scores = cosine_sim_cb[idx]
    cf_scores = cosine_sim_cf[idx]
    hybrid = alpha * cb_scores + (1 - alpha) * cf_scores
    hybrid[idx] = 0  # wyklucz sam siebie

    top_indices = np.argsort(hybrid)[::-1][:top_n]

    # SHAP — background = próbka losowych filmów (reprezentuje "brak cechy")
    background = shap.sample(features_array, shap_samples)
    predict_fn = lambda X: get_similarity_scores(X, idx, alpha)
    explainer = shap.KernelExplainer(predict_fn, background)

    results = []

    for rec_idx in top_indices:
        rec_title = movies["title"].iloc[rec_idx]
        score = float(hybrid[rec_idx])

        # Oblicz SHAP dla tego jednego kandydata
        shap_values = explainer.shap_values(features_array[rec_idx:rec_idx+1], silent=True)[0]

        # Top 3 cechy z największym pozytywnym wpływem
        shap_series = pd.Series(shap_values, index=feature_names)
        top_positive = shap_series[shap_series > 0].nlargest(3)
        top_negative = shap_series[shap_series < 0].nsmallest(2)

        # XAI — procentowy podział CB vs CF (jak w V1)
        contrib_cb = alpha * cb_scores[rec_idx]
        contrib_cf = (1 - alpha) * cf_scores[rec_idx]
        total = contrib_cb + contrib_cf
        pct_cb = (contrib_cb / total * 100) if total > 0 else 0
        pct_cf = (contrib_cf / total * 100) if total > 0 else 0

        # human_explanation generowany z danych SHAP
        if top_positive.empty:
            explanation = "Rekomendacja oparta głównie na ocenach społeczności."
        else:
            top_feature = top_positive.index[0].replace("genre_", "").replace("_", " ")
            if top_feature == "avg rating":
                explanation = "Wysoko oceniany przez widzów o podobnym guście."
            elif top_feature == "decade":
                explanation = f"Pochodzi z tej samej epoki co '{base_title}' — podobny klimat produkcji."
            else:
                explanation = f"Gatunek '{top_feature}' najbardziej zbliża ten film do '{base_title}'."

        result = {
            "movie_id": int(movies["movieId"].iloc[rec_idx]),
            "title": rec_title,
            "score": round(score, 4),
            "xai": {
                "content_contribution_pct": round(pct_cb, 1),
                "collaborative_contribution_pct": round(pct_cf, 1),
                "human_explanation": explanation,
                "shap_top_positive": top_positive.to_dict(),
                "shap_top_negative": top_negative.to_dict(),
            },
        }
        results.append(result)

        print(f"📽️  {rec_title}")
        print(f"   Score: {score:.4f}  |  CB: {pct_cb:.0f}%  CF: {pct_cf:.0f}%")
        print(f"   ↳ {explanation}")
        if not top_positive.empty:
            print(f"   Cechy wspierające:  {', '.join([f'{k.replace("genre_","")} (+{v:.3f})' for k, v in top_positive.items()])}")
        if not top_negative.empty:
            print(f"   Cechy osłabiające:  {', '.join([f'{k.replace("genre_","")} ({v:.3f})' for k, v in top_negative.items()])}")
        print()

    return results


# Test
results = explain_recommendation("Toy Story", alpha=0.5, top_n=3, shap_samples=50)

## 7. Wizualizacja SHAP

Interaktywny wykres pokazujący wpływ cech na rekomendację dla jednego wybranego kandydata.

In [ ]:
import matplotlib.pyplot as plt

TARGET = "Toy Story"
CANDIDATE = "Aladdin"  # ← zmień na dowolny film który chcesz zbadać
ALPHA = 0.5
SHAP_SAMPLES = 100

try:
    base_idx = movies[movies["title"].str.contains(TARGET, case=False, na=False, regex=False)].index[0]
    cand_idx = movies[movies["title"].str.contains(CANDIDATE, case=False, na=False, regex=False)].index[0]
except IndexError as e:
    print(f"Nie znaleziono filmu: {e}")
    base_idx = cand_idx = None

if base_idx is not None:
    background = shap.sample(features_array, SHAP_SAMPLES)
    predict_fn = lambda X: get_similarity_scores(X, base_idx, ALPHA)
    explainer = shap.KernelExplainer(predict_fn, background)

    shap_vals = explainer.shap_values(features_array[cand_idx:cand_idx+1], silent=True)

    # Filtrujemy tylko cechy z niezerowym wpływem — wykres czytelniejszy
    shap_series = pd.Series(shap_vals[0], index=feature_names)
    shap_nonzero = shap_series[shap_series.abs() > 0.001].sort_values()

    fig, ax = plt.subplots(figsize=(10, max(4, len(shap_nonzero) * 0.4)))
    colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in shap_nonzero.values]
    shap_nonzero.plot(kind="barh", ax=ax, color=colors)
    ax.axvline(0, color="white", linewidth=0.8)
    ax.set_title(f"SHAP — dlaczego '{movies['title'].iloc[cand_idx]}'\npasuje do '{movies['title'].iloc[base_idx]}'")
    ax.set_xlabel("Wartość SHAP (wpływ cechy na wynik)")
    ax.tick_params(axis="y", labelsize=9)
    plt.tight_layout()
    plt.show()

## 8. Weryfikacja kontraktu API

Upewniamy się że struktura wynikowa jest zgodna z istniejącym kontraktem (`MovieRecommendation` + `XAIExplanation`).

Pola wymagane przez API: `movie_id`, `title`, `poster_url`, `score`, `xai.content_contribution_pct`, `xai.collaborative_contribution_pct`, `xai.human_explanation`.

In [ ]:
REQUIRED_TOP = {"movie_id", "title", "poster_url", "score", "xai"}
REQUIRED_XAI = {"content_contribution_pct", "collaborative_contribution_pct", "human_explanation"}

# poster_url nie jest generowane w notatniku (to zadanie serwera) — dodajemy placeholder
if results:
    for r in results:
        r["poster_url"] = "https://via.placeholder.com/500x750"

    print("Weryfikacja struktury wyników:\n")
    all_ok = True
    for r in results:
        missing_top = REQUIRED_TOP - r.keys()
        missing_xai = REQUIRED_XAI - r["xai"].keys()
        if missing_top or missing_xai:
            print(f"❌ {r['title']}: brakuje {missing_top | missing_xai}")
            all_ok = False
        else:
            pct_sum = r["xai"]["content_contribution_pct"] + r["xai"]["collaborative_contribution_pct"]
            pct_ok = abs(pct_sum - 100) < 0.1
            status = "✅" if pct_ok else "⚠️  procenty nie sumują się do 100"
            print(f"{status} {r['title']} | score={r['score']} | CB={r['xai']['content_contribution_pct']}% CF={r['xai']['collaborative_contribution_pct']}%")
            print(f"      → {r['xai']['human_explanation']}")

    print()
    if all_ok:
        print("✅ Struktura zgodna z kontraktem API — gotowe do przepisania na recommender_engine.py v2.")
else:
    print("Brak wyników — uruchom najpierw komórkę z explain_recommendation().")

## 9. Ewaluacja — porównanie z baseline TF-IDF

Porównujemy RMSE obu modeli na zbiorze testowym. Niższe RMSE = model lepiej przewiduje rzeczywiste oceny.

> Komórka jest opcjonalna i może działać kilka minut.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

train_ratings, test_ratings = train_test_split(ratings, test_size=0.2, random_state=42)

user_item_train = train_ratings.pivot(index="movieId", columns="userId", values="rating").fillna(0)
user_item_train = user_item_train.reindex(movies["movieId"], fill_value=0)
cosine_sim_cf_train = cosine_similarity(user_item_train)


def predict_rating(user_id, movie_id, sim_cb, alpha=0.5):
    try:
        movie_idx = movies[movies["movieId"] == movie_id].index[0]
    except IndexError:
        return 3.0

    user_col = user_item_train.columns.get_loc(user_id) if user_id in user_item_train.columns else None
    if user_col is None:
        return 3.0

    user_ratings_vec = user_item_train.iloc[:, user_col].values
    rated_indices = np.where(user_ratings_vec > 0)[0]
    if len(rated_indices) == 0:
        return 3.0

    cb_sims = sim_cb[movie_idx]
    cf_sims = cosine_sim_cf_train[movie_idx]
    hybrid_sims = alpha * cb_sims + (1 - alpha) * cf_sims

    weights = hybrid_sims[rated_indices]
    actual = user_ratings_vec[rated_indices]
    sum_w = np.sum(np.abs(weights))
    return float(np.clip(np.sum(actual * weights) / sum_w, 0.5, 5.0)) if sum_w > 0 else 3.0


sample = test_ratings.sample(n=500, random_state=42)

true_r, pred_tfidf, pred_emb = [], [], []
for _, row in sample.iterrows():
    uid, mid, actual = int(row["userId"]), int(row["movieId"]), float(row["rating"])
    true_r.append(actual)
    pred_tfidf.append(predict_rating(uid, mid, cosine_sim_tfidf))
    pred_emb.append(predict_rating(uid, mid, cosine_sim_cb))

rmse_tfidf = mean_squared_error(true_r, pred_tfidf) ** 0.5
rmse_emb = mean_squared_error(true_r, pred_emb) ** 0.5

print("=" * 40)
print("EWALUACJA — RMSE (niżej = lepiej)")
print("=" * 40)
print(f"  TF-IDF baseline:   {rmse_tfidf:.4f}")
print(f"  Embeddingi (nowy): {rmse_emb:.4f}")
diff = rmse_tfidf - rmse_emb
print(f"  Różnica:           {diff:+.4f} ({'✅ nowy lepszy' if diff > 0 else '⚠️  baseline lepszy'})")